# Evaluating OTEL Traces with W&B Weave

This notebook walks through how to evaluate LLM traces already captured in Weave (instrumented via otel)

In [ ]:
import json, os, dotenv, openai, weave

dotenv.load_dotenv()

TEAM_NAME = "smle-demo"
PROJECT = "otel-evals"

client = weave.init(f"{TEAM_NAME}/{PROJECT}")

### Step 1 — Download trace dataset

Fetch `ChatCompletion` spans from the Weave project. Each call is one LLM request captured via OTEL.

In [ ]:
calls = list(client.get_calls(
    filter={"op_names": [f"weave:///{TEAM_NAME}/{PROJECT}/op/chatcompletion:*"]}, #filter applied to extract only chatcompletion spans
))
print(f"{len(calls)} calls fetched")

### Step 2 — Extract inputs and outputs

Convert calls to a DataFrame — mirrors how Phoenix's `get_spans_dataframe()` works.

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "id":     c.id,
    "input":  " ".join(m["content"] for m in c.inputs["input.value"]["messages"] if m.get("content")),
    "output": ((c.output or {}).get("output.value") or {}).get("choices", [{}])[0].get("message", {}).get("content", ""),
} for c in calls]).set_index("id")

rows = df.reset_index().to_dict("records")
print(f"{len(df)} rows extracted")
df[["input", "output"]].head(3)

### Step 3 — Publish as a Weave Dataset

In [ ]:
dataset = weave.Dataset(name="otel_traces_dataset", rows=rows)
weave.publish(dataset)
print(f"Published dataset with {len(dataset.rows)} rows")

### Step 4 — Define scorers

Two scorers run against each trace output:

- **Sentiment** — LLM-as-judge using `gpt-4o-mini` to label positive / negative / neutral
- **Response length** — word count with a 0–1 adequacy score

In [ ]:
@weave.op(name="sentiment")
def sentiment_scorer(output: str):
    resp = openai.OpenAI().chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": 'Respond ONLY with JSON: {"label": "positive"|"negative"|"neutral", "score": 0-1}'},
            {"role": "user",   "content": output[:4000]},
        ],
        max_tokens=60,
    )
    data = json.loads(resp.choices[0].message.content)
    return {"label": data["label"], "score": float(data["score"])}


@weave.op(name="response_length")
def length_scorer(output: str):
    words = len(output.split())
    score = min(1.0, words / 50.0) if words >= 10 else 0.2 * (words / 10.0)
    return {"word_count": words, "score": round(score, 4)}

### Step 5 — Run evaluation

The passthrough model returns the trace output as-is — no new LLM calls, we're scoring what was already captured.

In [ ]:
@weave.op()
def passthrough_model(output: str):
    return output

evaluation = weave.Evaluation(
    dataset=dataset,
    scorers=[sentiment_scorer, length_scorer],
    evaluation_name="otel_traces_eval",
)

result = await evaluation.evaluate(passthrough_model)
print(result)